# 🟢 Impact Strategy — Colab Launcher

One tab to run everything: install deps, load your keys from **Colab Secrets**, test the paper trader interactively, and (optionally) run the research notebook's analytics.

### First-time setup (2 min)
1. Click the **🔑 key icon** in the left sidebar (Secrets).
2. Add these, toggling *Notebook access* ON for each:
   - `ALPACA_API_KEY` — your Alpaca **paper** key
   - `ALPACA_SECRET_KEY` — your Alpaca **paper** secret
   - `SLACK_WEBHOOK_URL` — *(optional)* for desk pings
   - `EMAIL_TO`, `EMAIL_FROM`, `EMAIL_APP_PASSWORD` — *(optional)* email summaries
3. Run the cells top to bottom.

> **Important:** Colab only runs while this tab is open — it does **not** schedule itself. Use Colab to *test and watch* the trader. For it to fire daily while you're at work with the tab closed, use the **GitHub Action** in `SETUP_paper_trader.md`. Same code, runs on a schedule.

*Paper trading only. Research tooling, not investment advice.*

In [ ]:
#@title Install dependencies
!pip install -q alpaca-py yfinance pandas numpy
print("Dependencies installed.")

In [ ]:
#@title Load secrets from Colab → environment
import os
try:
    from google.colab import userdata
    def _get(k):
        try: return userdata.get(k)
        except Exception: return ""
    for k in ["ALPACA_API_KEY","ALPACA_SECRET_KEY","SLACK_WEBHOOK_URL",
              "EMAIL_TO","EMAIL_FROM","EMAIL_APP_PASSWORD"]:
        v = _get(k)
        if v: os.environ[k] = v
    print("Loaded:", [k for k in ["ALPACA_API_KEY","ALPACA_SECRET_KEY","SLACK_WEBHOOK_URL",
              "EMAIL_TO","EMAIL_FROM","EMAIL_APP_PASSWORD"] if os.environ.get(k)])
except ImportError:
    print("Not in Colab — set env vars yourself.")

# Safety defaults for interactive testing
os.environ["DRY_RUN"] = "true"        # logs intended orders, submits nothing
os.environ["ALWAYS_NOTIFY"] = "true"  # send a summary even on no-trade days
print("DRY_RUN =", os.environ["DRY_RUN"], "(flip in the cell below when ready)")

## Get the trader

Upload `auto_paper_trader.py` to this Colab session (📁 Files → upload), **or** paste it into a cell. Then run the trader once and read the output. With `DRY_RUN=true` it will print the exact orders it *would* place and send you a test summary — but submit nothing.

In [ ]:
#@title Run the paper trader once (dry run)
# Make sure auto_paper_trader.py is uploaded to this session first.
import importlib.util, os
assert os.path.exists("auto_paper_trader.py"), "Upload auto_paper_trader.py via the Files panel first."
spec = importlib.util.spec_from_file_location("apt", "auto_paper_trader.py")
apt = importlib.util.module_from_spec(spec); spec.loader.exec_module(apt)
apt.run()   # dry run — reads your secrets, checks market, logs intended trades, pings you

### When you're ready to place *paper* orders

After you've watched a dry run or two and trust the output, flip the switch and run again. This still only touches your **paper** account — the client is hard-locked to `paper=True`.

In [ ]:
#@title Go live on PAPER (still fake money)
import os, importlib
os.environ["DRY_RUN"] = "false"
import importlib.util
spec = importlib.util.spec_from_file_location("apt", "auto_paper_trader.py")
apt = importlib.util.module_from_spec(spec); spec.loader.exec_module(apt)
apt.run()   # now submits PAPER orders and sends the real summary

## (Optional) Run the research notebook here too

You can open `impact_quant_strategy.ipynb` directly in Colab (**File → Upload notebook**) to run the backtest, today's picks, action grades, and the Section 9 diagnostics. It's self-contained — just uncomment its `pip install` cell.

### Recap of the safe path
1. **Dry run** here until the logs look right.
2. **Paper orders** here (or via the scheduled GitHub Action for hands-off daily runs).
3. **~90 days / 100+ trades**, then compare to buy-and-hold and re-check IC + t-stat in Section 9.
4. Only if it clears the bar do we discuss real money — tiny size, hard caps, kill switch.

*Paper trading only. Not investment advice. No promise of returns.*